In [ ]:
%%capture
!pip install awswrangler
!pip install polars

In [ ]:
%load_ext autoreload
%autoreload 2
from IPython.core.display import HTML
display(HTML("<style>pre { white-space: pre !important; }</style>"))
%config Completer.use_jedi = False
import awswrangler as wr
import sys
sys.path.append("../")
import pandas as pd
import polars as pl
import numpy as np
from datetime import datetime, timedelta
import calendar
from pytz import timezone

PATH_DATA = 's3://ada-us-east-1-sbx-live-mx-bmin-data/MI36597/'
tz = timezone('America/Mexico_city')
today  =datetime.now(tz=tz).date().strftime('%Y-%m-%d')

In [ ]:
# PATH_DATA

## funciones aux

In [ ]:
# toolbox athena
def run_query_athena(query:str, database:str='mx_master', verbose:bool=True,engine='pandas', **read_csv_kwargs)-> pd.DataFrame:
    """Run athena query
        
        Parameters
        ----------
        query : str, sql query
        database : str, defatult 'mx_master'.
                    Name of the database.
        Verbose : bool, defatul True.
                    If True shows data scanned of query in Kb and csv location in s3.
        read_csv_kwargs : Options to pass to read_csv pandas method.

        Returns
        ----------
        Pandas Dataframe with the result of the query.
    """
    query_execution_id = wr.athena.start_query_execution(sql=query, 
                                                         database=database, 
                                                         workgroup="sandbox")
    query_execution = wr.athena.wait_query(query_execution_id)
    data_scanned_kb = query_execution['Statistics']['DataScannedInBytes']/1000
    result_location = query_execution['ResultConfiguration']['OutputLocation']
    # leer con pandas o wr, veamos...
    if engine=='pandas':
        result_df = pd.read_csv(result_location,**read_csv_kwargs)
    elif engine=='polars':
        result_df = pl.scan_csv(result_location, infer_schema_length=20000,**read_csv_kwargs)
    if verbose:
        print(F"Data scanned {data_scanned_kb} kb.")
        print(f'Query result at: {result_location}')
    return result_df

def show_max_partitions_athena(table_name:str, database:str='mx_master')-> dict:
    """Returns the name of the partition column and it's maximum value.
        
        Parameters
        ----------
        table_name : str, name of the table. Should belong to database
        database : str, defatult 'mx_master'.
                    Name of the database.
    """
    qry = f"SHOW PARTITIONS {database}.{table_name};"
    temp = run_query_athena(qry,verbose=True,sep='|',header=None)
    partdf = (temp
              .sort_values(by=temp.columns.tolist(),ascending=False)
              .reset_index(drop=True)
             )
    laux = partdf[0].str.split('/')[0]
    final_dict=dict()
    for i in laux:
        splitted = i.split('=')
        final_dict[splitted[0]] = splitted[1]
    # partdf = partdf[0].str.split('/',expand=True)
    return final_dict

In [ ]:
hoy = datetime.now(tz=tz).date()
print(hoy)
# Monday = 0, Tuesday = 1, ..., Sunday = 6
dia_semana = hoy.weekday()

if dia_semana == 0:
    # Si corre lunes, toma viernes, sábado y domingo
    fecha_inicio = hoy - timedelta(days=3)
    fecha_fin = hoy - timedelta(days=1)
else:
    # Martes a viernes, toma solo el día anterior
    fecha_inicio = hoy - timedelta(days=1)
    fecha_fin = hoy - timedelta(days=1)

fecha_inicio = fecha_inicio.strftime("%Y-%m-%d")
fecha_fin = fecha_fin.strftime("%Y-%m-%d")

print("fecha_inicio:", fecha_inicio)
print("fecha_fin:", fecha_fin)

In [ ]:
from time import time,sleep
start = time()

## parametros

In [ ]:
estados_automarket = ['DF','EM','MO','PU']
estados_query = (',').join([f"'{x}'" for x in estados_automarket])
monto_low = 200000
monto_high= 500000

## carga

In [ ]:
max_fecha_estados = show_max_partitions_athena("t_mdco_tla543_per_cte_basico_sem")['load_date']

In [ ]:
query_ingresos =f"""WITH ingresos AS (
    SELECT DISTINCT
        customer_platform_id AS NU_CTE,
        first_name AS NOMBRE,
        middle_name AS SEGUNDO_NOMBRE,
        last_name AS AP_PATERNO,
        second_last_name AS AP_MATERNO,
        cell_phone_number_id AS CELULAR,
        customer_state_name AS ESTADO,
        substr(product_request_id, -8, 8) AS FOLIO,
        date_format(CAST(registry_entry_date AS timestamp), '%d/%m/%Y') AS FECHA_DE_INGRESO,
        car_credit_req_fnl_decsn_desc AS DECISION_SISTEMA,
        requested_amount AS MONTO_SOLICITADO,
        substr(vehicle_financing_plan_desc, 1, 50) AS PAQUETE,
        atev_age_number AS EDAD,
        vehicle_value_amount AS VALOR_VEHICULO,
        vehicle_year_id AS MODELO,
        vehicle_condition_type AS TIPO_VEHICULO,
        loan_initial_payment_amount AS MONTO_ENGANCHE,
        cr_request_max_term_number AS PLAZO,
        user_country_nationality_name AS NACIONALIDAD,
        gender_type AS SEXO,
        veh_loan_init_pymt_per AS ENGANCHE,
        atev_car_credit_int_rate_per AS TASA_SIN_IVA,
        vehicle_supplier_commission_per AS COMISION,
        source_channel_id AS CANAL
    FROM "mx_master"."t_msan_profit_base"
    WHERE source_channel_id IN ('GLOMO_2', 'GLOMO NP')
      AND registry_entry_date BETWEEN DATE('{fecha_inicio}') AND DATE('{fecha_fin}')
      AND regexp_like(car_credit_req_fnl_decsn_desc, '(?i)ACEPTADO|APROBADO|VIABLE')
),

ingresos_2 AS (
    SELECT
        NU_CTE,
        NOMBRE,
        SEGUNDO_NOMBRE,
        AP_PATERNO,
        AP_MATERNO,
        CELULAR,
        ESTADO,
        FOLIO,
        FECHA_DE_INGRESO,
        DECISION_SISTEMA,
        MONTO_SOLICITADO,
        PAQUETE,
        EDAD,
        VALOR_VEHICULO,
        MODELO,
        TIPO_VEHICULO,
        MONTO_ENGANCHE,
        PLAZO,
        NACIONALIDAD,
        SEXO,
        ENGANCHE,
        TASA_SIN_IVA,
        COMISION,
        CANAL
    FROM (
        SELECT
            ingresos.*,
            row_number() OVER (
                PARTITION BY NU_CTE
                ORDER BY MONTO_SOLICITADO DESC
            ) AS row_num
        FROM ingresos
    )
    WHERE row_num = 1
),
estados AS (
select customer_id,state_id
FROM "mx_master"."t_mdco_tla543_per_cte_basico_sem"
WHERE load_date = date('{max_fecha_estados}')
)

SELECT x.*, y.state_id
FROM ingresos_2 x
LEFT JOIN estados y
ON x.NU_CTE = y.customer_id
where y.state_id IN ({estados_query})

"""

In [ ]:
ingresos = run_query_athena(query_ingresos,engine='polars')

In [ ]:
max_cutoff_date_br = show_max_partitions_athena('t_mbtq_remote_wallet_vector')['cutoff_date']
query_banca_remota = f"""
 SELECT DISTINCT customer_id AS NU_CTE
    FROM "mx_master"."t_mbtq_remote_wallet_vector"
    WHERE cutoff_date = DATE('{max_cutoff_date_br}')
"""

In [ ]:
banca_remota = run_query_athena(query_banca_remota,engine='polars')

In [ ]:
query_prodesk = f"""
    SELECT DISTINCT
        substr(product_request_id, -8, 8) AS FOLIO
    FROM "mx_master"."t_msan_hir_fin_pro_i_atm_cr"
    WHERE CAST(car_cr_task_end_time_date AS date) BETWEEN DATE('{fecha_inicio}') AND DATE('{fecha_fin}')

"""

In [ ]:
prodesk = run_query_athena(query_prodesk,engine='polars')

In [ ]:
query_exclusiones = f"""
WITH excepciones_flag AS (
    SELECT
        *,
        CASE
            WHEN condusef_reus_mark_type = 1 THEN 'CONDUSEF_GLOBAL'
            WHEN condusef_reus1_mark_type = 1 THEN 'PDP'
            WHEN national_foreign_type = 1 THEN 'RESIDENTE_EXT'
            WHEN politically_exposed_type = 1 THEN 'POL_EXPUESTO'
            WHEN advisors_id = 1 THEN 'CONSEJEROS'
            WHEN undesirable_customer_mark_type = 1 THEN 'INDESEABLE'
            WHEN demised_customer_mark_type = 1 THEN 'FALLECIDO'
            WHEN fraud_transaction_type = 1 THEN 'FRAUDE'
            WHEN transfer_sold_debt_mark_type = 1 THEN 'VENDIDA'
            WHEN employee_type IS NOT NULL THEN 'EMPLEADO_BBVA'
            WHEN image_id = 1 THEN 'IMAGEN'
            WHEN minor_influence_cust_mark_type = 1 THEN 'IMAGEN_MENOR'
            WHEN under_age_mark_type = 1 THEN 'MENOR_EDAD'
            WHEN credit_bankruptcy_mark_type = 1 THEN 'QUEBRANTOS'
            ELSE NULL
        END AS EXCLUSION_REASON
    FROM "mx_master"."t_mdco_tla539_per_cte_excepcion_pyt"
),

exclusiones AS (
    SELECT DISTINCT
        customer_id AS NU_CTE,
        EXCLUSION_REASON
    FROM excepciones_flag
    WHERE EXCLUSION_REASON IS NOT NULL
)

SELECT *
FROM exclusiones;
"""

In [ ]:
exclusiones = run_query_athena(query_exclusiones,engine='polars')

In [ ]:
query_nomina = f"""
WITH nomina_excluir AS (
    SELECT DISTINCT
        customer_id AS NU_CTE,
        1 as MARCO_EMPLEADOS
    FROM "mx_master"."t_mdco_tla938_stock_nom_pyt"
    WHERE customer_id IS NOT NULL
      AND dpst_3000_payrl_acct_clsfn_type = 'PS'
      AND company_customer_id IN ('4205081122', '4205078083')
      AND (
            COALESCE(m1_end_payroll_amount, 0)
          + COALESCE(m2_end_payroll_amount, 0)
      ) > 0
      AND duplicate_customer_type IS NULL
      AND cutoff_date >= date_add(
            'month',
            -3,
            date_trunc('month', current_date)
      )
)

SELECT *
FROM nomina_excluir;
"""

In [ ]:
nomina = run_query_athena(query_nomina,engine='polars')

In [ ]:
# max_fecha_query_motor = 
max_alt_date_motor = show_max_partitions_athena('t_mdco_tla1922_mda_motor_alter')['alternative_register_date']

In [ ]:
query_motor = f"""
SELECT customer_id AS NU_CTE,
customer_commercial_segment_type,
1 as excluir_m
FROM "mx_master"."t_mdco_tla1922_mda_motor_alter"
WHERE alternative_register_date = DATE('{max_alt_date_motor}')
AND customer_commercial_segment_type IN ('P0','P1','P2','P3')
"""

In [ ]:
excluir_motor = run_query_athena(query_motor,engine='polars')

## Proceso y filtros

In [ ]:
select_cols = [
    "rank_automarket",
    "FOLIO",
    "NU_CTE",
    "NOMBRE",
    "AP_PATERNO",
    "AP_MATERNO",
    "ESTADO",
    "MONTO_SOLICITADO",
    "VALOR_VEHICULO",
    "TIPO_VEHICULO",
    "TASA_SIN_IVA"
]

In [ ]:
final = (ingresos
 .with_columns(pl.col('FOLIO').cast(pl.String))
 .join(prodesk, on='FOLIO',how='anti')
 .join(banca_remota, on='NU_CTE',how='anti')
 .filter((pl.col('TIPO_VEHICULO')=='SEMINUEVO')
         # &(pl.col('ESTADO').str.strip_chars()=='DISTRITO FEDERAL')
         &(pl.col('MONTO_SOLICITADO').is_between(monto_low,monto_high))
        )
 .sort(['NU_CTE'],descending = [False])
 .with_columns(pl.col('NU_CTE').rank(method = 'dense',descending=False).alias('rank_automarket'))
 .filter(pl.col('rank_automarket')<=20)
 .select(select_cols)
 .join(exclusiones,on='NU_CTE',how='left')
 .filter(pl.col('EXCLUSION_REASON').is_null())
 .drop(['EXCLUSION_REASON'])
 .join(nomina,on='NU_CTE',how='left')
 .filter(pl.col('MARCO_EMPLEADOS').is_null())
 .drop(['MARCO_EMPLEADOS','rank_automarket'])
 .join(excluir_motor,on='NU_CTE',how='left')
 .filter(pl.col('excluir_m').is_null())
 .drop(['excluir_m','customer_commercial_segment_type'])
 .collect().to_pandas()
 .assign(fecha_ejecucion = datetime.now(tz=tz).strftime("%Y-%m-%d %H:%M"))
)

In [ ]:
final

In [ ]:
# final

In [ ]:
end = time()
print(f'Tardó {round(end-start,2)} segundos.')

## Write

In [ ]:
path = f'{PATH_DATA}piloto_edas/{today}/piloto_edas_{today}.csv'
# final.to_csv(path,index=False)
print(path)

In [ ]:
today

In [ ]:
!aws s3 cp ./pilotoEdas.ipynb s3://ada-us-east-1-sbx-live-mx-bmin-data/MI36597/piloto_edas/notebooks/pilotoEdas.ipynb